In [3]:
from pyspark.sql import SparkSession

print("STEP 1: Starting SparkSession builder")

spark = SparkSession.builder \
    .appName("StreamFromKafkaToParquet") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:8333") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.12.0"
    ) \
    .config("spark.hadoop.fs.s3a.access.key", "dev_key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dev_secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    ) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")\
    .config("spark.sql.catalog.silver", "org.apache.iceberg.spark.SparkCatalog")\
    .config("spark.sql.catalog.silver.type", "hive")\
    .config("spark.sql.catalog.silver.uri", "thrift://localhost:9084")\
    .config("spark.sql.catalog.silver.warehouse", "s3a://silver/")\
    .getOrCreate()

print("STEP 2: SparkSession created")

print("STEP 3: Listing namespaces")
spark.sql("SHOW NAMESPACES").show(truncate=False)

print("STEP 4: Finished")

STEP 1: Starting SparkSession builder
STEP 2: SparkSession created
STEP 3: Listing namespaces
+---------+
|namespace|
+---------+
|default  |
+---------+

STEP 4: Finished


In [2]:
df = spark.read.parquet(
    "s3a://warehouse/transaction/event_date=20749"
)

df.show(truncate=False)
print("Total rows:", df.count())

26/10/27 23:28:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------+----------+-------+--------+--------------------+-------+--------+
|transaction_id|account_id|amount |currency|event_time          |country|status  |
+--------------+----------+-------+--------+--------------------+-------+--------+
|txn-000009    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000005    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000003    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000011    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000004    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000008    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000010    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000006    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn-000007    |acct-0048 |2400.00|USD     |2026-10-23T10:15:32Z|ID     |APPROVED|
|txn

In [10]:
# spark.sql("drop namespace if exists iceberg.pipeline_test").show()
# spark.sql("drop table if exists iceberg.silver.transactions purge").show(truncate=False)
# spark.sql("drop table if exists iceberg.pipeline_test.smoke_test").show(truncate=False)
spark.sql("SELECT * FROM silver.datalake.transactions").show(truncate=False)

+--------------+----------+-------+--------+-------------------+-------+--------+
|transaction_id|account_id|amount |currency|event_time         |country|status  |
+--------------+----------+-------+--------+-------------------+-------+--------+
|txn-000002    |acct-0048 |2400.00|USD     |2026-10-23 17:15:32|ID     |APPROVED|
|txn-000001    |acct-0048 |2400.00|USD     |2026-10-23 17:15:32|ID     |APPROVED|
|txn-000003    |acct-0048 |2400.00|USD     |2026-10-23 17:15:32|ID     |APPROVED|
+--------------+----------+-------+--------+-------------------+-------+--------+

